In [ ]:
# Mini-project: Advanced Statistical Analysis of Apple Inc. Stock Data


# 👩‍🏫 👩🏿‍🏫 What You’ll learn
# Master statistical analysis of financial data using NumPy and SciPy.
# Learn effective data visualization techniques with Matplotlib for financial trends.
# Apply hypothesis testing to financial datasets for meaningful insights.
# Understand and utilize advanced statistical techniques in NumPy and SciPy.


# Project Description
# Using the AAPL (Apple Inc.) stock dataset, conduct the following analyses:



# Initial Data Exploration
# Load the dataset using Pandas. Check for null values and understand data types.
# Examine the time series properties of the data (e.g., frequency, trends).


# Data Visualization
# Utilize Matplotlib to plot closing prices and traded volume over time.
# Create a candlestick chart to depict high and low prices.


# Statistical Analysis
# Compute summary statistics (mean, median, standard deviation) for key columns.
# Analyze closing prices with a moving average.


# Hypothesis Testing
# Execute a t-test to compare average closing prices across different years.
# Examine daily returns’ distribution and test for normality using SciPy.


# Advanced Statistical Techniques (Bonus)
# Statistical Functions in NumPy: Employ NumPy’s statistical functions for in-depth stock data analysis.
# E.g., Use convolve for moving averages, or np.corrcoef to explore correlations between financial metrics.
# Analyze correlations between moving averages of closing prices and trading volume across time periods.




In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, signal
from scipy.stats import ttest_ind, shapiro
import mplfinance as mpf

plt.style.use('default')


## 1. Data Loading and Exploration

In [ ]:
# Load dataset
df = pd.read_csv('data/Apple Stock Prices (1981 to 2023).csv')

# Convert Date column
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Sort by date
df = df.sort_values('Date')

print("Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

df.head()


In [ ]:
# Time series properties
df = df.set_index('Date')

print("Date range:", df.index.min(), "to", df.index.max())
print("Frequency inference:", pd.infer_freq(df.index[:100]))


## 2. Data Visualization

In [ ]:
# Closing price over time
plt.figure(figsize=(12,5))
plt.plot(df.index, df['Close'])
plt.title('Apple Closing Price Over Time')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.show()


In [ ]:
# Trading volume over time
plt.figure(figsize=(12,5))
plt.plot(df.index, df['Volume'])
plt.title('Apple Trading Volume Over Time')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.show()


In [ ]:
# Candlestick chart (recent 200 days for readability)
candles = df[['Open','High','Low','Close','Volume']].tail(200)

mpf.plot(
    candles,
    type='candle',
    volume=True,
    style='yahoo',
    title='Apple Candlestick Chart (Last 200 Trading Days)'
)


## 3. Statistical Analysis

In [ ]:
# Summary statistics
summary_stats = df[['Open','High','Low','Close','Adj Close','Volume']].agg(
    ['mean','median','std']
)
summary_stats


In [ ]:
# Moving average using pandas
df['MA_50'] = df['Close'].rolling(window=50).mean()

plt.figure(figsize=(12,5))
plt.plot(df.index, df['Close'], label='Close')
plt.plot(df.index, df['MA_50'], label='50-Day MA')
plt.title('Closing Price vs 50-Day Moving Average')
plt.legend()
plt.show()


## 4. Hypothesis Testing

In [ ]:
# Compare average closing prices between two years

df['Year'] = df.index.year

year1 = 2021
year2 = 2022

close_year1 = df[df['Year']==year1]['Close']
close_year2 = df[df['Year']==year2]['Close']

t_stat, p_value = ttest_ind(close_year1, close_year2, equal_var=False)

print(f'T-statistic: {t_stat:.4f}')
print(f'P-value: {p_value:.4f}')

if p_value < 0.05:
    print("Reject H0: Average closing prices are significantly different.")
else:
    print("Fail to reject H0.")


In [ ]:
# Daily returns
df['Daily_Return'] = df['Close'].pct_change()
returns = df['Daily_Return'].dropna()

# Distribution
plt.figure(figsize=(10,5))
plt.hist(returns, bins=50)
plt.title('Distribution of Daily Returns')
plt.show()

# Normality test
sample_returns = returns.sample(min(5000, len(returns)), random_state=42)

stat, p = shapiro(sample_returns)

print("Shapiro Statistic:", stat)
print("P-value:", p)

if p < 0.05:
    print("Daily returns are NOT normally distributed.")
else:
    print("Daily returns appear normally distributed.")


## 5. Advanced Statistical Techniques (Bonus)

In [ ]:
# Moving average using NumPy convolve

window = 50
weights = np.ones(window) / window

ma_convolve = np.convolve(df['Close'], weights, mode='valid')

plt.figure(figsize=(12,5))
plt.plot(df.index[window-1:], ma_convolve)
plt.title('Moving Average Using NumPy Convolve')
plt.show()


In [ ]:
# Correlation analysis

valid = df[['Volume','MA_50']].dropna()

corr_matrix = np.corrcoef(valid['Volume'], valid['MA_50'])

print("Correlation Matrix")
print(corr_matrix)


In [ ]:
# Signal processing using SciPy

smoothed_close = signal.savgol_filter(df['Close'], 51, 3)

plt.figure(figsize=(12,5))
plt.plot(df.index, df['Close'], label='Original')
plt.plot(df.index, smoothed_close, label='Smoothed')
plt.title('Savitzky-Golay Smoothing')
plt.legend()
plt.show()
